[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brilliantbeaver/alexpose/blob/main/experiments/sjepa/gavd6-pm/new_nb_09_00_methodology_and_contract.ipynb)

# new_nb_09_00. Methodology and contract for the equivariance-coupled retrain

**Notebook 1 of 4, Idea 9 arm 2.** The four notebooks are one continuous experiment and they are
meant to be read in order: fix the decision rule before any data has been seen, prove the training
term is capable of doing what it claims, run the real training ladder, then apply the rule that was
fixed in step one.

**What this notebook answers.** Four things that are easy to bend after seeing results: what question
is being asked, what will be measured, what will count as success, and what data the answer is
allowed to rest on. This notebook trains nothing and produces no result.

**What it assumes you already have.** The locked canonical pose cohort on disk, the completed
baseline curriculum checkpoint from `04_pretrain_sjepa_on_normal.ipynb`, and no knowledge of the
outcome. Familiarity with `nb_05a` (Idea 5) and `nb_09a` (Idea 9 arm 1) helps, but section 1 restates
what this series needs from them.

**What it produces.** One file, `idea9_arm2_contract.json`, holding the question, the endpoints, the
guardrails, the credit rule, the registered seed list, and a verified description of the cohort with
hashes.

**What comes next.** `new_nb_09_01` spends seconds on synthetic fixtures to check that the training
term can move the endpoint at all. That check is the gate the expensive notebook has to clear before
it is allowed to run.

Where the series comes from: section 8 of `nb_09b_equivariant_retrain.ipynb` describes an arm 2 run
that was never launched. It tells the reader to paste the antisymmetric head, the anatomical mirror,
and the equivariance loss into `04_pretrain_sjepa_on_normal.ipynb`, then run a multi-seed 600-epoch
curriculum and re-score every checkpoint. This series carries out those four steps as its own
executable path and leaves notebook 04 untouched, so the baseline lineage and the experiment stay
separable.

**Research use only.** Folder labels such as stroke and parkinsons are dataset annotations, not
clinical diagnoses. The source video is the independent unit of evidence throughout. Every number
this series will eventually report is transductive: the encoder saw every evaluation sequence during
training.


## 0. What the four notebooks do, and why there are four of them

The split is not organisational tidiness. Each boundary closes off a specific way of fooling
yourself, and each notebook hands the next one a settled fact.

1. **This notebook, the contract.** It writes down the question, the endpoint, and the decision rule
   before any number exists. What it hands forward is a rule that can no longer be tuned, because it
   is already on disk with a timestamp.
2. **`new_nb_09_01`, the mechanism check.** Four checks that run in seconds on synthetic fixtures. A
   badly written equivariance term is silently a no-op, so these have to pass before any real compute
   is spent. What it hands forward is one specific, tested form of the loss.
3. **`new_nb_09_02`, the real ladder.** It trains the full five-stage curriculum twice per seed, once
   with the term switched off and once with it on, and reports what happened. What it hands forward is
   checkpoints, and nothing else: it deliberately draws no conclusion.
4. **`new_nb_09_03`, the verdict.** It recomputes every endpoint from the saved checkpoints and
   applies this notebook's rule mechanically, condition by condition.

**Why preregistration, and why it is a practice rather than paperwork.** Writing the rule down first
is not an administrative formality. It is a defence against a specific and very ordinary failure. Once
you can see a result, every threshold looks negotiable in a direction that happens to favour that
result, and the renegotiation feels like good judgement while it is happening rather than like
cheating. Fixing the rule in a file before the data exists is what allows a later verdict, especially
a negative one, to be evidence instead of an opinion.

That matters here more than usual, because the last notebook in this series does end in a negative
verdict while the headline measurement moves by a large margin. The only reason that combination is
readable as an honest finding rather than as a grudging concession is that its conditions, including
the one that fails, were all written on this page first.

The notebooks share no kernel state. Everything that crosses a boundary crosses it as a file on disk,
which is what makes the series resumable and independently checkable.

**The cell below** resolves the project root, loads the environment, and creates the arm 2 output
directory. Read its output to confirm two things before going further: that `mode` is `real`, so the
paths point at the real artifact tree rather than a smoke sandbox, and that `arm-2 output dir` ends in
`idea9_arm2`, so nothing this series writes can collide with a baseline artifact name.


In [1]:
from pathlib import Path
import os, sys, json, math, hashlib, copy, time, warnings

import numpy as np
import pandas as pd

RANDOM_SEED = 42


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "experiments" / "sjepa" / "gavd6-pm"
try:
    from dotenv import load_dotenv
    load_dotenv(TUTORIAL_DIR / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

TRUTHY = {"1", "true", "yes", "on"}
REQUESTED_MODE = os.getenv("GAVD_MODE", "smoke").strip().lower()
INCLUDE_AUGMENTED = os.getenv("SJEPA_INCLUDE_AUGMENTED_NORMAL", "0").strip().lower() in TRUTHY
ARTIFACT_ROOT = Path(os.getenv("GAVD_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")).expanduser()
CACHE_DIR = Path(os.getenv("GAVD_CACHE_DIR", TUTORIAL_DIR / "work" / "cache")).expanduser()
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

CONDITIONS = ["normal", "parkinsons", "stroke", "myopathic", "cerebralpalsy"]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]

# The six authorised left/right pairs the head, the signed target, and rho are all built from.
LEFT_RIGHT_PAIRS = [(11, 12), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]
# The full 16-pair anatomical mirror, matching notebook 04's geometric_view flip.
FULL_MIRROR_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10),
    (11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32),
]

# Arm-2 output lives in its own directory. Nothing here ever writes a baseline artifact name.
ARM2_DIR_NAME = "idea9_arm2"
BASELINE_CHECKPOINT_NAME = (
    "sjepa_curriculum_final_augmented.pt" if INCLUDE_AUGMENTED else "sjepa_curriculum_final.pt"
)


def artifact_dir_for(mode):
    return ARTIFACT_ROOT / mode


MODE = REQUESTED_MODE
ARTIFACT_DIR = artifact_dir_for(MODE)
ARM2_DIR = ARTIFACT_DIR / ARM2_DIR_NAME
ARM2_DIR.mkdir(parents=True, exist_ok=True)

print(f"project root      : {PROJECT_ROOT}")
print(f"mode              : {MODE}")
print(f"artifact root     : {ARTIFACT_ROOT}")
print(f"arm-2 output dir  : {ARM2_DIR}")
print(f"augmented normals : {INCLUDE_AUGMENTED}")
print(f"baseline checkpoint name: {BASELINE_CHECKPOINT_NAME}")

project root      : /Users/pmui/dev/alexpose
mode              : real
artifact root     : /Users/pmui/dev/alexpose/experiments/sjepa/gavd6/work/artifacts
arm-2 output dir  : /Users/pmui/dev/alexpose/experiments/sjepa/gavd6/work/artifacts/real/idea9_arm2
augmented normals : True
baseline checkpoint name: sjepa_curriculum_final_augmented.pt


## 1. Step 1 of 6. The question, stated so that it can fail

**What we are doing in this step.** Turning a vague ambition, "make the encoder respect the mirror",
into a sentence whose answer is allowed to be no.

**Why this step comes first.** Two earlier experiments set arm 2 up, and the chain matters, because
arm 2 is not a fresh start. It is an attempt to close the one escape route the earlier two left open.

- **Idea 5, `nb_05a`.** A ridge probe read a signed left-minus-right laterality axis out of the frozen
  encoder on source-disjoint folds. Verdict: `INFORMATIVE NULL`. The measurement was valid and the
  answer was no. On this cohort the frozen representation does not make a signed laterality axis
  linearly available above a raw-coordinate baseline, and it does not even reach an untrained-encoder
  floor.
- **Idea 9 arm 1, `nb_09a`.** The objection was that Idea 5's readout might have had the wrong shape,
  since a signed quantity ought to be read by a head that is antisymmetric by construction. Arm 1
  built exactly that head and verified its wiring at a slope of exactly -1, so the head really was
  antisymmetric and this was not an implementation bug. The null survived, and something worse
  happened: a mirror-symmetrized lane, which is mathematically blind to left and right, outscored the
  antisymmetric treatment. Verdict: `ARTIFACT (side-agnostic nuisance control fired)`. That withdraws
  the claim rather than answering it, which is a weaker epistemic position than a clean null, not a
  stronger one.

Both experiments froze the encoder. Neither could therefore test the more interesting possibility:
that the encoder never learned to treat a body and its mirror image as sign-flipped versions of each
other, and that asking it to do so during training would change the representation. That is the
escape route arm 2 exists to close.

**The question, as registered:**

> Does adding a label-free equivariance term to the curriculum objective make the trained encoder
> measurably more mirror-honest than the identical recipe without that term, by more than the
> run-to-run variation across seeds, and without degrading the representation on the tasks the encoder
> is used for?

Three clauses in that sentence are load-bearing, and each one is a way for the answer to come out no.
"More mirror-honest" needs a measurement, which section 3 defines. "By more than the run-to-run
variation across seeds" needs a control trained at the same seeds, which section 6 sets up. "Without
degrading the representation" needs guardrails, which section 4 registers.

**Definition, used everywhere from here on.** The **anatomical mirror** `M` does two things at once,
and both halves are required. It reflects the body geometrically, negating the sideways coordinate,
and it swaps the identities of every left and right landmark pair, so the left-knee slot receives the
reflected right knee. A geometric reflection on its own is not the anatomical mirror, because it would
leave a left-labelled slot holding right-side motion, and a model could then "respect" it by learning
nothing about sides at all.

**The term being added.** With `s` for the antisymmetric head and `x` for a raw skeleton sequence,

```
L_equiv = mean( ( s(encoder(Mx)) + s(encoder(x)) )^2 )
```

and the total objective becomes `L_JEPA + 0.05 * L_VICReg + 0.25 * L_group + w * L_equiv`, with
`w = 0` for the control rung and `w = 0.02` for the treatment rung. Two of those names are fixed by
this project and are worth stating precisely: `L_VICReg` is only the label-free invariance,
variance-hinge, and covariance regularizer computed on projected student features, and `L_group` is a
separate label-aware term equal to within-condition compactness plus a centroid-margin penalty.
Nothing else differs between the two rungs.

**Two properties of `L_equiv` that decide how its result can be read.** It uses no label of any kind,
not a condition folder and not a source video, so it cannot manufacture a transductive win by
memorising which videos happen to be lateralised. And it constrains the encoder rather than the
readout, which is precisely what Idea 5 and arm 1 could not do.

**What we may not conclude yet.** Nothing at all, and one specific caution is worth flagging now. The
form of `L_equiv` written above is the form registered here, and `new_nb_09_01` will show that this
exact form can be satisfied without changing the encoder at all. The repair, and the evidence that a
repair was needed, are recorded there, on synthetic fixtures, before any real compute is spent.


## 2. Step 2 of 6. The one correctness trap, and why it is a trap

**What we are doing in this step.** Naming, before any code runs, the single mistake that would make
this whole experiment measure nothing while appearing to work perfectly.

**Why it comes here.** Because it is a mistake about the mirror argument, not about the training
setup, so it has to be understood before the endpoint or the decision rule will make sense.

The antisymmetric head is antisymmetric *by construction*. It computes a pure difference over left and
right landmark pairs, so swapping its own inputs negates its output exactly, for every input and every
parameter value. That is arithmetic, not learning. Which means a loss written as

```
mean( ( s(swap of the head's own tokens) + s(tokens) )^2 )
```

is identically zero. Not approximately zero and not small: algebraically zero, with a gradient of zero
everywhere. Written that way the term trains nothing at all, while looking entirely reasonable in code
and producing a loss curve that sits convincingly at zero. A reader watching that curve would conclude
the constraint had been satisfied immediately, when in fact it was never imposed.

The fix is to apply the mirror to the raw skeleton and run both the original and the reflection through
the encoder. The encoder is not equivariant by construction, so the residual is genuinely nonzero and
its gradient reaches encoder weights.

**Why this is stated as a claim to be tested rather than as an argument.** The reasoning above is
convincing, and convincing reasoning is exactly what produced the bug in the first place, in
`nb_09b`. So `new_nb_09_01` measures both halves instead of asserting them: that the head-only version
delivers exactly zero loss and exactly zero gradient, and that the through-the-encoder version delivers
nonzero gradient into encoder weights.

The swap identity is still worth keeping, as a wiring self-check on the head. It simply cannot be the
training loss.


## 3. Step 3 of 6. Why the primary endpoint is not R-squared

**What we are doing in this step.** Choosing what to measure, and rejecting the obvious choice for a
reason that was recorded before arm 2 began.

**The obvious choice, and why it is wrong here.** The natural way to score arm 2 would be to re-run
arm 1's probe on each retrained checkpoint and compare held-out-source R-squared. Arm 1's own bundle
says why that would be a mistake.

Arm 1 pre-registered a y-quality gate: the signed target must carry at least 30 percent of its
variance *between* source videos, because source-disjoint folds hold out whole videos, and if almost
all of the target's variance lies *within* videos then a held-out-source R-squared is scoring noise. On
the real cohort the between-source fraction came out at **0.0747** against the required **0.30**. The
gate failed. Consistently with that, every learned lane came out negative: the antisymmetric head
reached -0.206, the untrained-encoder floor -0.027, and the standard encoder comparator -0.602. A
difference between two negative numbers, both of which the item's own preregistered gate declares
uninterpretable, cannot support a conclusion in either direction.

This is the direct causal link from arm 1 to arm 2. Arm 2 does not abandon R-squared because R-squared
was inconvenient. It abandons R-squared because arm 1 measured, in advance and against a threshold it
had committed to, that this cohort cannot support a labelled held-out-source R-squared at all. The
binding constraint is the cohort, which has 18 independent source videos, and not the model or the
readout.

**The replacement endpoint.** So the primary endpoint is a quantity that needs no target variance
whatsoever: the normalised mirror-consistency residual, written **rho**. Write `T` for the
antisymmetric contraction with the identity feature map, which is the head with all of its parameters
removed, and recall `M`, the anatomical mirror that both reflects the body and swaps left and right
landmark identities. Then

```
rho(encoder) = mean_seq || T(enc(Mx)) + T(enc(x)) ||^2
             / mean_seq ( 0.5 * ( || T(enc(x)) ||^2 + || T(enc(Mx)) ||^2 ) )
```

**The scale of rho, stated once and used everywhere.** Lower is better.

- **rho = 0** means the encoder represents the mirrored body as the exact sign flip of the original
  along this axis. This is perfect mirror equivariance and it is the best possible value.
- **rho = 2** means the two representations are unrelated along this axis.
- **rho = 4** means the encoder is fully mirror-symmetric, so it cannot distinguish a body from its
  reflection at all. This is complete mirror blindness and it is the worst possible value.

**Why rho is usable here at all, which is the whole point.** rho is **label-free**: no condition
folder and no source video enters it, so it cannot be inflated by the encoder having memorised which
videos are lateralised. And rho is **parameter-free**: there is nothing to fit, no regularisation
constant to choose, and no fold structure, so the same instrument reads every checkpoint identically
and the comparison across rungs is exact rather than approximate. Those two properties together are
the entire reason a quantity like this can stand in for the endpoint whose quality gate failed. It is
also the quantity `L_equiv` pressures, with the learned feature map replaced by the identity, so a
trained head cannot flatter its own arm of the experiment.

rho is computed per sequence and reported per source video, so the comparison between rungs can be
paired by video.

**What rho is not, stated now so it cannot drift later.** rho is a symmetry property of the
representation. It is not accuracy, not class separation, and not clinical value. An improvement in rho
is not by itself evidence of any downstream benefit.

**Whether the scale above is real.** It is asserted here and it is *verified* in `new_nb_09_01`, on
encoders built by hand so that the right answer is known in advance. Nothing in this notebook should be
believed about rho's scale until that check has run.


## 4. Step 4 of 6. The full pre-registered measurement plan

**What we are about to do.** Commit, in a file, to every endpoint, every guardrail, the exact decision
rule, and the seed list. The next cell writes that commitment into `PRE_REGISTERED` and prints the two
parts that decide the verdict.

**Why now, before anything is trained.** Because after training there will be numbers, and a rule
chosen in the presence of numbers is not a rule.

### The endpoints

**Primary.** rho on the **target encoder**, which is the frozen EMA teacher that every downstream
readout in this project actually reads. Reported per source video across the canonical cohort.

**Secondary.** rho on the **view encoder**, which is the network `L_equiv` optimises directly.
Reporting both separates two different questions: whether the pressure worked at all, and whether it
survived the EMA transfer into the artifact that gets used.

**Secondary.** The **measured** anatomical-mirror slope through the encoder. Idea 5 measured -0.741 for
the baseline and arm 1 measured -0.223 with its own head. If the equivariance term works, this slope
should move toward -1. It is a measured number throughout and is never asserted to be -1, which is the
distinction arm 1's wiring check makes necessary: a head-only token swap gives exactly -1 by
construction and therefore says nothing about the encoder.

**Tertiary, always reported beside the failed gate.** Arm 1's six-lane source-disjoint ladder,
recomputed per checkpoint, so the record is complete even though arm 1's own y-quality gate says these
numbers are weak evidence.

### The guardrails, and why registering them in advance is the most important thing on this page

A treatment rung could improve rho by making the representation worse. If the encoder simply had less
to be inconsistent about, rho would fall for a reason that has nothing to do with learning mirror
structure. Three guardrails are registered against that:

- `feature_std`, the mean feature standard deviation of the frozen target encoder's pooled embeddings.
  Higher is safer.
- `mean_pair_cosine`, the mean pairwise cosine between those embeddings. Lower is safer, because
  everything pointing the same way is collapse.
- `source_grouped_five_class_balanced_accuracy`, a five-class condition probe on the frozen target
  encoder, scored on source-grouped folds. Higher is safer.

Here is the teaching point, and it is the reason this page exists. **A guardrail registered after the
fact is not evidence; it is a rebuttal invented to fit an outcome.** If the guardrails were chosen once
the endpoint result was visible, then a reader would have every right to ask whether they had been
picked because they failed, and no answer would be available. Because they are named here, before any
rung has run, a later guardrail failure is a prediction that came true and a later clean sweep is a
prediction that came true. Symmetrically, they can no longer be quietly dropped if they turn out to be
inconvenient. That symmetry is the whole mechanism by which a "no credit" verdict later in this series
becomes credible rather than arbitrary.

Note also what a guardrail is *for*. Its job is to catch destruction, not to demonstrate benefit. A
guardrail is allowed to be a weak test. What is not allowed is reading its value as a result.

### The credit rule

The equivariance term earns credit only if **all three** of the following hold. This is recorded in the
contract as `all_three_required: true`, and it means two passes and a failure is not a partial success.

1. **Condition 1, the improvement exceeds the control's seed spread.** The mean improvement in rho from
   the control rung to the treatment rung must be larger than the control rung's own seed-to-seed
   standard deviation. The logic is a trajectory control rather than a population claim: if switching
   the term on moves the endpoint by less than merely changing the random seed does, then nothing has
   been demonstrated.
2. **Condition 2, the paired-by-source bootstrap interval excludes zero.** Resampling *source videos*,
   not sequences, because windows cut from one video are not independent of each other, and pairing by
   source because the two rungs of a seed differ in exactly one term.
3. **Condition 3, no guardrail regression.** No guardrail may fall by more than the control rung's own
   seed spread for that guardrail. This is what stops the endpoint from being bought with
   representation quality.

Any other outcome is reported as **no credit**, with the numbers stated plainly.

### The seeds

Registered seeds are **0, 1, 2, 3 and 4**, five in total, for both rungs. The seed list is part of the
rule and not a detail, because the control rung's spread across those seeds is the yardstick condition
1 is measured against.

**What to look at in the next cell's output.** Two JSON blocks. In the first, confirm
`parameter_free: true`, `label_free: true` and `requires_fitting: false` on the primary endpoint, since
those are the properties section 3 argued make rho admissible, and confirm that `why_not_r2` records
arm 1's failed gate at 0.0747 against 0.30. In the second, confirm `all_three_required: true`.

**What we may conclude from it.** Only that the rule now exists on disk. It says nothing about the
result, because no result exists.


In [2]:
PRE_REGISTERED = {
    "question": (
        "Does a label-free equivariance term make the trained encoder measurably more mirror-honest than "
        "the identical recipe without it, by more than seed-to-seed variation, without degrading the "
        "representation?"
    ),
    "L_equiv": "mean((s(enc(Mx)) + s(enc(x)))^2), M = anatomical mirror on RAW coords, run THROUGH the "
               "view encoder. Label-free. A head-only token swap is identically zero and trains nothing.",
    "total_loss": "L_JEPA + 0.05*L_VICReg + 0.25*L_group + w*L_equiv, w = 0 for D0 and 0.02 for E1",
    "primary_endpoint": {
        "name": "rho_target_encoder",
        "definition": "mean_seq||T(enc(Mx)) + T(enc(x))||^2 / mean_seq(0.5*(||T(enc(x))||^2 + "
                      "||T(enc(Mx))||^2)), T = antisymmetric contraction with the identity feature map",
        "scale": "0 = exact sign flip (best), 2 = unrelated, 4 = fully symmetric (worst)",
        "why_not_r2": "Arm 1's y-quality gate FAILED on this cohort (between-source fraction 0.0747 "
                      "against a 0.30 threshold), so held-out-source R-squared is uninterpretable here.",
        "parameter_free": True, "label_free": True, "requires_fitting": False,
    },
    "secondary_endpoints": ["rho_view_encoder", "measured_anatomical_mirror_slope"],
    "tertiary_endpoints": ["arm1_six_lane_source_disjoint_r2 (reported beside the failed y-gate)"],
    "guardrails": ["feature_std", "mean_pair_cosine", "source_grouped_five_class_balanced_accuracy"],
    "credit_rule": {
        "1_exceeds_seed_spread": "mean(rho_D0) - mean(rho_E1) > std(rho_D0) across seeds",
        "2_paired_bootstrap": "paired-by-source bootstrap CI for the improvement excludes zero",
        "3_no_guardrail_regression": "no guardrail falls by more than the D0 seed spread",
        "all_three_required": True,
    },
    "ladder": {"rungs": ["D0", "E1"], "seeds": [0, 1, 2, 3, 4],
               "equiv_weight": {"D0": 0.0, "E1": 0.02}},
    "framing": [
        "All results are TRANSDUCTIVE: the encoder saw every evaluation sequence during training.",
        "The source video is the independent unit of evidence.",
        "Folder labels are dataset annotations, not clinical diagnoses.",
    ],
}
print(json.dumps(PRE_REGISTERED["primary_endpoint"], indent=2))
print()
print(json.dumps(PRE_REGISTERED["credit_rule"], indent=2))

{
  "name": "rho_target_encoder",
  "definition": "mean_seq||T(enc(Mx)) + T(enc(x))||^2 / mean_seq(0.5*(||T(enc(x))||^2 + ||T(enc(Mx))||^2)), T = antisymmetric contraction with the identity feature map",
  "scale": "0 = exact sign flip (best), 2 = unrelated, 4 = fully symmetric (worst)",
  "why_not_r2": "Arm 1's y-quality gate FAILED on this cohort (between-source fraction 0.0747 against a 0.30 threshold), so held-out-source R-squared is uninterpretable here.",
  "parameter_free": true,
  "label_free": true,
  "requires_fitting": false
}

{
  "1_exceeds_seed_spread": "mean(rho_D0) - mean(rho_E1) > std(rho_D0) across seeds",
  "2_paired_bootstrap": "paired-by-source bootstrap CI for the improvement excludes zero",
  "3_no_guardrail_regression": "no guardrail falls by more than the D0 seed spread",
  "all_three_required": true
}


## 5. Step 5 of 6. The data contract

**What we are about to do.** Verify every input the ladder will depend on, before any compute is
committed to it.

**Why this step comes before training rather than after.** A cohort mismatch discovered after the
ladder has run is not a nuisance, it is a discarded ladder. And an unverified cohort is worse than a
mismatched one, because the results would still be reported.

The ladder trains on the same corpus the baseline used, so that the two lineages are comparable, and it
is evaluated on the canonical subset, matching arm 1's `canonical_subset_only` flag. Concretely,
training uses the locked canonical cohort plus the opt-in augmentation-normal pool because
`SJEPA_INCLUDE_AUGMENTED_NORMAL` is enabled, and evaluation uses the canonical 96 sequences only. That
asymmetry is deliberate: the extra normal sequences help the curriculum and would muddy a comparison
against arm 1's numbers, which were computed on the canonical subset.

The cell below checks, in order: that the pose cache matches the locked cohort identity by identity;
that the mask whitelist is the de-duplicated 12-point MS-PD set; that the mapping file exists, with its
sha256 recorded; and that the baseline checkpoint is present, readable, built at the expected width and
depth, and marked as a completed curriculum. It records the baseline checkpoint's sha256 so that
`new_nb_09_03` can prove this experiment never modified it.

**What to look at in the output.** The condition counts, which should read cerebralpalsy 16,
myopathic 47, normal 12, parkinsons 9, stroke 12, summing to 96 canonical sequences from 18 source
videos. The training corpus line, 159 sequences from 35 source videos once the augmentation-normal pool
is included. And the baseline fingerprint, whose first 16 characters are `ea59fea055f0230b`: that is
the shared fingerprint of the checkpoint Idea 5 and arm 1 were both computed on, so seeing it here is
what makes this series continuous with them.

**What we may conclude.** That the inputs are what the contract says they are. Note the number that is
about to matter later: **18** source videos, and only one of them contributing all 12 canonical normal
sequences. Both facts are cohort properties, visible now, long before any result exists.


In [3]:
def interpolate_low_visibility(sequence, threshold=0.45, max_gap=4):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Expected [T, 33, 4], received {sequence.shape}")
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    filled = valid.copy()
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        for left, right in zip(observed[:-1], observed[1:]):
            gap = int(right - left - 1)
            if not 0 < gap <= max_gap:
                continue
            fraction = (np.arange(1, gap + 1, dtype=np.float32) / (gap + 1))[:, None]
            sequence[left + 1:right, joint, :3] = (
                sequence[left, joint, :3][None, :] * (1.0 - fraction)
                + sequence[right, joint, :3][None, :] * fraction)
            filled[left + 1:right, joint] = True
        sequence[~filled[:, joint], joint, :3] = np.nan
    sequence[..., 3] = visibility
    return sequence, valid


def center_and_scale(sequence, eps=1e-6):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    xyz = sequence[..., :3]
    left_hip, right_hip = xyz[:, 23], xyz[:, 24]
    left_ok = np.isfinite(left_hip).all(axis=1)
    right_ok = np.isfinite(right_hip).all(axis=1)
    pelvis = np.full((len(xyz), 3), np.nan, dtype=np.float32)
    both = left_ok & right_ok
    pelvis[both] = 0.5 * (left_hip[both] + right_hip[both])
    pelvis[left_ok & ~right_ok] = left_hip[left_ok & ~right_ok]
    pelvis[right_ok & ~left_ok] = right_hip[right_ok & ~left_ok]
    pelvis_ok = np.isfinite(pelvis).all(axis=1)
    fallback = np.median(pelvis[pelvis_ok], axis=0) if pelvis_ok.any() else np.zeros(3)
    pelvis[~np.isfinite(pelvis).all(axis=1)] = fallback
    xyz = xyz - pelvis[:, None, :]
    shoulder_width = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hip_width = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    body_scale = np.nanmedian(np.maximum(shoulder_width, hip_width))
    if not np.isfinite(body_scale) or body_scale < eps:
        body_scale = 1.0
    sequence[..., :3] = np.nan_to_num(xyz / body_scale, nan=0.0, posinf=0.0, neginf=0.0)
    return np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_resize(array, frames):
    array = np.asarray(array)
    if len(array) == frames:
        return array.copy()
    if len(array) < 2:
        return np.repeat(array, frames, axis=0)
    old_t = np.linspace(0.0, 1.0, len(array))
    new_t = np.linspace(0.0, 1.0, frames)
    flat = array.reshape(len(array), -1)
    resized = np.stack([np.interp(new_t, old_t, flat[:, i]) for i in range(flat.shape[1])], axis=1)
    return resized.reshape(frames, *array.shape[1:])


def prepare_sequence(raw_sequence, frames):
    """raw_sequence: [T, 33, 4] -> (xyz [frames, 33, 3] float32, valid [frames, 33] bool)."""
    interpolated, valid = interpolate_low_visibility(raw_sequence)
    scaled = center_and_scale(interpolated)
    xyz = temporal_resize(scaled[..., :3], frames)
    valid_resized = temporal_resize(valid.astype(np.float32), frames) > 0.5
    return xyz.astype(np.float32), valid_resized


def anatomical_mirror_raw(coords, pairs=FULL_MIRROR_PAIRS):
    """Mirror the RAW [T, 33, 4] column so the SAME preprocessing can be re-run on the reflection."""
    mirrored = np.asarray(coords, dtype=np.float32).copy()
    mirrored[:, :, 0] = -mirrored[:, :, 0]
    for left, right in pairs:
        mirrored[:, [left, right], :] = mirrored[:, [right, left], :]
    return mirrored


def signed_left_minus_right(coords):
    """Item 05's frozen target: signed per-side excursion, left minus right, on preprocessed coords."""
    coords = np.asarray(coords, dtype=np.float64)[..., :3]
    total = 0.0
    for left, right in LEFT_RIGHT_PAIRS:
        total += coords[:, left, :].std(axis=0).sum() - coords[:, right, :].std(axis=0).sum()
    return float(total)

POSE_CACHE_CONTRACT = json.loads((TUTORIAL_DIR / "pose_cache_contract.json").read_text())
COMPATIBLE_EXTRACTION_VERSIONS = frozenset(POSE_CACHE_CONTRACT["compatible_extraction_versions"])
COHORT_ROOT = PROJECT_ROOT / "data" / "gavd"
EXPECTED_SEQUENCE_COUNTS = {"normal": 12, "parkinsons": 9, "stroke": 12,
                            "myopathic": 47, "cerebralpalsy": 16}
EXPECTED_SEQUENCE_IDS = {
    condition: {path.stem for path in (COHORT_ROOT / condition).glob("*.csv")}
    for condition in CONDITIONS
}
MIN_AUGMENTED_NEURO_OBSERVED = 0.45


def canonical_pose_records(pose_dir, conditions=CONDITIONS):
    """Load the locked canonical pose cache with the same identity checks notebook 04 applies."""
    records = []
    for condition in conditions:
        folder = Path(pose_dir) / condition
        available = {path.stem: path for path in folder.glob("*.npz")}
        expected = EXPECTED_SEQUENCE_IDS[condition]
        if set(available) != expected:
            raise ValueError(
                f"Canonical {condition} pose cache does not match the locked cohort at {folder}: "
                f"expected {len(expected)}, found {len(available)}")
        for sequence_id in sorted(expected):
            data = np.load(available[sequence_id], allow_pickle=False)
            version = str(data["extraction_version"].item())
            if version not in COMPATIBLE_EXTRACTION_VERSIONS:
                raise ValueError(f"Unsupported extraction version {version} in {available[sequence_id]}")
            sequence = data["sequence"].astype(np.float32)
            if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
                raise ValueError(f"Bad pose shape {sequence.shape}")
            records.append({
                "condition": condition,
                "sequence_id": str(data["sequence_id"].item()),
                "video_id": str(data["video_id"].item()),
                "raw": sequence,
                "cohort": "canonical",
            })
    return records


def augmented_normal_records(artifact_dir):
    """Load the opt-in augmentation-normal cohort through its selection report, as notebook 04 does."""
    if not INCLUDE_AUGMENTED:
        return []
    folder = artifact_dir / "poses_augmented" / "normal"
    report_path = artifact_dir / "augmented_pose_extraction_report.csv"
    if not folder.exists() or not report_path.is_file():
        raise FileNotFoundError(
            f"SJEPA_INCLUDE_AUGMENTED_NORMAL is enabled but {folder} or {report_path} is missing. "
            "Run notes/migrate_augmented_pose_artifacts.py, or notes/annotate_normal_clips.py then "
            "notes/extract_augmented_poses.py.")
    report = pd.read_csv(report_path)
    candidates = report[~report["status"].astype(str).str.startswith("error")].copy()
    accepted = pd.to_numeric(candidates["neuro_observed"], errors="coerce") >= MIN_AUGMENTED_NEURO_OBSERVED
    selected = set(candidates.loc[accepted, "sequence_id"].astype(str))
    available = {path.stem: path for path in folder.glob("*.npz")}
    if set(available) != selected:
        raise ValueError("Augmented pose cache does not match its selection report")
    records = []
    for sequence_id in sorted(selected):
        data = np.load(available[sequence_id], allow_pickle=False)
        records.append({
            "condition": "normal",
            "sequence_id": str(data["sequence_id"].item()),
            "video_id": str(data["video_id"].item()),
            "raw": data["sequence"].astype(np.float32),
            "cohort": "augmented_normal",
        })
    return records

import torch

MAPPING_RELATIVE_PATH = Path("experiments/multiple-sclerosis/mapping-data/ms-pd-mapping.md")
MAPPING_PATH = PROJECT_ROOT / MAPPING_RELATIVE_PATH
if not MAPPING_PATH.is_file():
    raise FileNotFoundError(f"Required mapping file is missing: {MAPPING_PATH}")
MAPPING_SHA256 = hashlib.sha256(MAPPING_PATH.read_bytes()).hexdigest()

MASK_KEYPOINT_NAMES = ["LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HIP", "RIGHT_HIP",
                       "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
                       "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX"]
if MASK_KEYPOINTS != sorted(set(MASK_KEYPOINTS)) or len(MASK_KEYPOINTS) != 12:
    raise ValueError("The mask whitelist must be the de-duplicated 12-point MS-PD set")

POSE_DIR = ARTIFACT_DIR / "poses"
canonical = canonical_pose_records(POSE_DIR)
augmented = augmented_normal_records(ARTIFACT_DIR)

canonical_meta = pd.DataFrame([{k: r[k] for k in ("condition", "sequence_id", "video_id", "cohort")}
                               for r in canonical])
training_meta = pd.DataFrame([{k: r[k] for k in ("condition", "sequence_id", "video_id", "cohort")}
                              for r in canonical + augmented])

print("canonical cohort by condition:")
print(canonical_meta.groupby("condition").size().to_string())
print(f"\ncanonical      : {len(canonical)} sequences from "
      f"{canonical_meta['video_id'].nunique()} source videos")
print(f"augmented normal: {len(augmented)} sequences from "
      f"{pd.DataFrame(augmented)['video_id'].nunique() if augmented else 0} source videos")
print(f"training corpus : {len(canonical) + len(augmented)} sequences from "
      f"{training_meta['video_id'].nunique()} source videos")

baseline_path = ARTIFACT_DIR / BASELINE_CHECKPOINT_NAME
if not baseline_path.is_file():
    raise FileNotFoundError(f"Baseline checkpoint is missing: {baseline_path}")
BASELINE_SHA256 = hashlib.sha256(baseline_path.read_bytes()).hexdigest()
baseline_meta = torch.load(baseline_path, map_location="cpu", weights_only=False)
BASELINE_FINGERPRINT = str(baseline_meta["dataset_fingerprint"])
print(f"\nbaseline checkpoint : {baseline_path.name}")
print(f"baseline fingerprint: {BASELINE_FINGERPRINT[:16]}")
print(f"baseline sha256     : {BASELINE_SHA256[:16]}")
print(f"baseline config     : {baseline_meta['config']}")
if baseline_meta["mask_keypoints"] != MASK_KEYPOINTS:
    raise ValueError("Baseline checkpoint mask whitelist does not match the 12-point set")
if not baseline_meta.get("curriculum_complete", False):
    raise ValueError("Baseline checkpoint is not a completed curriculum final")

canonical cohort by condition:
condition
cerebralpalsy    16
myopathic        47
normal           12
parkinsons        9
stroke           12

canonical      : 96 sequences from 18 source videos
augmented normal: 63 sequences from 17 source videos
training corpus : 159 sequences from 35 source videos

baseline checkpoint : sjepa_curriculum_final_augmented.pt
baseline fingerprint: ea59fea055f0230b
baseline sha256     : 3c39ab36d4be8dd9
baseline config     : {'frames': 64, 'joints': 33, 'coordinate_dim': 3, 'segment_length': 4, 'embed_dim': 96, 'encoder_depth': 4, 'predictor_depth': 2, 'heads': 4}


## 6. Step 6 of 6. What is deliberately being reproduced, and what is deliberately new

**What we are about to do.** Write the whole contract, including the cohort description and the
hashes, to `idea9_arm2_contract.json`.

**Why the control rung is not the existing baseline checkpoint.** This is the most easily missed design
decision in the series, so it is worth being explicit about what escape route it closes *before* any
of its numbers appear. The existing baseline is one draw from a distribution of training trajectories.
Comparing the treatment rung against that single draw would confound the equivariance term with
ordinary run-to-run variation, and there would be no way afterwards to separate the two. So the control
rung is the same recipe rerun from a fresh seed with the equivariance term switched off, at the same
seeds as the treatment. That turns run-to-run variation from an unknown into a measured quantity, which
is exactly what condition 1 of the credit rule needs.

The existing baseline checkpoint still appears in the results, as a **reference row**, so a reader can
see where the published lineage sits relative to the ladder. It is never the control.

**Three things that are new relative to notebook 04, each a fix rather than a preference.**

- The seed is threaded through the model initialisation, the batch sampler, and the mask sampler.
  Notebook 04 hardcodes 42 in the per-stage generator and in the fingerprint payload, so copying it
  verbatim would have made five nominal seeds produce five identical runs, a seed spread of exactly
  zero, and a condition 1 that any difference at all would clear.
- `seed`, `equiv_weight` and `equiv_on` join the fingerprint payload, so each rung has its own identity
  and no rung can be mistaken for the baseline lineage.
- Each rung persists its own checkpoint and JSON, so the ladder resumes after an interruption instead
  of restarting.

**What to look at in the output.** The written path, which must be inside `idea9_arm2/`, and the cohort
block: 159 training sequences from 35 sources, 96 canonical sequences from 18 sources, the five
condition counts, and `evaluation_subset` recording that scoring happens on the canonical subset only.

**What we may conclude.** That the contract is now a file, and that from this point on the rule is
readable by anyone auditing the series without their having to trust this notebook's prose.


In [4]:
CONTRACT = {
    "notebook": "new_nb_09_00_methodology_and_contract",
    "series": "new_nb_09",
    "arm": "arm2_equivariance_coupled_retrain",
    "supersedes": "the smoke-configured idea9_equivariant_retrain_result.json, which was written with "
                  "mode=real by nb_09b while training with SMOKE_CONFIG",
    "mode": MODE,
    "pre_registered": PRE_REGISTERED,
    "cohort": {
        "training_sequences": len(canonical) + len(augmented),
        "training_sources": int(training_meta["video_id"].nunique()),
        "canonical_sequences": len(canonical),
        "canonical_sources": int(canonical_meta["video_id"].nunique()),
        "canonical_condition_counts": canonical_meta.groupby("condition").size().to_dict(),
        "include_augmented_normal": INCLUDE_AUGMENTED,
        "evaluation_subset": "canonical only, matching Arm 1's canonical_subset_only flag",
    },
    "contract": {
        "mask_keypoints": MASK_KEYPOINTS,
        "mask_keypoint_names": MASK_KEYPOINT_NAMES,
        "mapping_path": str(MAPPING_RELATIVE_PATH),
        "mapping_sha256": MAPPING_SHA256,
        "pose_extraction_versions_accepted": sorted(COMPATIBLE_EXTRACTION_VERSIONS),
        "baseline_checkpoint": baseline_path.name,
        "baseline_fingerprint": BASELINE_FINGERPRINT,
        "baseline_sha256": BASELINE_SHA256,
        "baseline_role": "reference row only; never the control rung",
    },
    "arm1_reference": {
        "verdict": "ARTIFACT (side-agnostic nuisance control fired)",
        "y_between_source_fraction": 0.07466055304332204,
        "y_gate_threshold": 0.30,
        "y_gate_passed": False,
        "A_prime_r2": -0.20574789599382415,
        "C_floor_r2": -0.027031078336773096,
        "D_standard_r2": -0.6021929001488207,
        "wiring_swap_slope": -1.0000000000000002,
        "measured_anatomical_mirror_slope": -0.22290579837876792,
    },
    "item05_reference": {
        "verdict": "INFORMATIVE NULL",
        "A_learned_r2": -0.6021929001488207,
        "C_floor_r2": -0.1561506987225285,
        "sign_consistency": 0.4444444444444444,
        "measured_anatomical_mirror_slope": -0.7408177852917672,
        "note": "Some older notes cite -0.343 for this slope. The current bundle on disk says -0.741, "
                "and that is the number this series uses.",
    },
}
contract_path = ARM2_DIR / "idea9_arm2_contract.json"
contract_path.write_text(json.dumps(CONTRACT, indent=2, sort_keys=False), encoding="utf-8")
print(f"wrote {contract_path}")
print(json.dumps(CONTRACT["cohort"], indent=2))

wrote /Users/pmui/dev/alexpose/experiments/sjepa/gavd6/work/artifacts/real/idea9_arm2/idea9_arm2_contract.json
{
  "training_sequences": 159,
  "training_sources": 35,
  "canonical_sequences": 96,
  "canonical_sources": 18,
  "canonical_condition_counts": {
    "cerebralpalsy": 16,
    "myopathic": 47,
    "normal": 12,
    "parkinsons": 9,
    "stroke": 12
  },
  "include_augmented_normal": true,
  "evaluation_subset": "canonical only, matching Arm 1's canonical_subset_only flag"
}


## 7. What this notebook has established, and what comes next

**Established.**

1. The question is written down in a form that can be answered no, and it is placed in its lineage:
   Idea 5 returned an informative null, arm 1 withdrew its own claim as an artifact, and arm 2 exists to
   close the one route neither of them could test, namely that the encoder was never asked to respect
   the mirror.
2. The primary endpoint is rho, the normalised mirror-consistency residual, on the frozen EMA target
   encoder. It is label-free and parameter-free, and its scale runs from 0, exact mirror equivariance
   and the best value, through 2, unrelated, to 4, complete mirror blindness and the worst value.
3. The endpoint is not a held-out-source R-squared, and the reason is arm 1's own preregistered
   y-quality gate, which failed at a between-source variance fraction of 0.0747 against a required
   0.30. That makes a labelled held-out-source R-squared uninterpretable on this cohort in either
   direction.
4. The decision rule is fixed with all three of its conditions required: the improvement must exceed
   the control's seed spread, the paired-by-source bootstrap interval must exclude zero, and no
   registered guardrail may fall by more than the control's seed spread. The guardrails are named:
   `feature_std`, `mean_pair_cosine`, and a source-grouped five-class balanced accuracy.
5. The data contract is verified against the cohort on disk rather than assumed, and the baseline
   checkpoint's sha256 is recorded so its integrity can be proven later.

**Not established.** Nothing has been measured. No number on this page is a result, and the
equivariance term has not yet been shown to be capable of changing an encoder at all.

**Two pointers for a reader of the finished series,** so that this page is not mistaken for the whole
story. The registration above is left exactly as it was written, including in two places where the
completed experiment departed from it. The registered form of `L_equiv` turns out to be satisfiable
without changing the encoder, and `new_nb_09_01` measures that, repairs it, and records both. The
registered seed list of five turns out to be three, because the ladder ran **3 seeds against the 5
registered here**, and `new_nb_09_03` records the shortfall as a protocol deviation together with its
effect on inference. Neither departure is edited into this
contract, because a contract that is quietly updated to match the outcome is not a contract.

**Next.** `new_nb_09_01` proves the mechanism on a small controlled problem: whether the equivariance
term reaches encoder weights at all, whether rho really reads 0 and 4 at the two ends of its scale, and
whether the loss can be satisfied without the encoder changing. That is the gate the real ladder has to
clear before it is worth running.
